# Simple test of implementing PID control for the robot to stay upright

In [5]:
# Import standard libraries
from pathlib import Path
import time

# Import third-party libraries
from IPython.display import clear_output
import mujoco
import mujoco.viewer

In [6]:
# Settings
print(Path.cwd())
MJCF_PATH = Path("/workspace/hardware/bala2-fire-simplified.xml")
MOTOR_SPEED_LIMIT = 1.0    # Max motor speed in each direction
PRINT_EVERY = 50           # Number of sim loop iterations before printing sensor readings

# Actuator names (from MJCF file)
LEFT_MOTOR = "left_motor"
RIGHT_MOTOR = "right_motor"

# Sensor names (from MJCF file)
IMU_ACCEL = "imu_accel"
IMU_GYRO = "imu_gyro"
IMU_ORIENTATION = "imu_orientation"
LEFT_WHEEL_POS = "left_wheel_pos"
RIGHT_WHEEL_POS = "right_wheel_pos"
LEFT_WHEEL_VEL = "left_wheel_vel"
RIGHT_WHEEL_VEL = "right_wheel_vel"

/workspace/software


In [17]:
class PIDController:
    """
    A simple PID (in fact PSD) controller for balancing the robot.
    """
    def __init__(self, kp, ki, kd):
        self.kp = kp
        self.ki = ki
        self.kd = kd
        self.integral = 0.0
        self.prev_error = 0.0

    def update(self, error, error_rate, dt):
        self.integral += error * dt
        derivative = error_rate
        output = self.kp * error + self.ki * self.integral + self.kd * derivative
        self.prev_error = error
        return output

In [18]:
# Load model into MuJoCo
model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))

# Use model to get the simulation state
data  = mujoco.MjData(model)

In [19]:
# Get ID of actuators from MJCF names
left_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, LEFT_MOTOR)
right_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, RIGHT_MOTOR)

# Print IDs
print(f"Left motor ID: {left_motor_id}")
print(f"Right motor ID: {right_motor_id}")

Left motor ID: 0
Right motor ID: 1


In [33]:
pid = PIDController(kp=5.0, ki=0.1, kd=0.5)  # Example PID gains

In [34]:
# Resets simulation data to defaults
mujoco.mj_resetData(model, data)

# Launch MuJoCo simulator and GUI
steps = 0
with mujoco.viewer.launch_passive(model, data) as viewer:
    # Define free-look camera (control with mouse), looking at robot's back-right
    viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    viewer.cam.lookat[:] = [0, 0, 0.05]
    viewer.cam.distance  = 0.8 
    viewer.cam.azimuth   = 45
    viewer.cam.elevation = -25
    pitch = 0.0
    tipped = False # checks if the robot has tipped over, then resets the simulation
    # Simulation loop
    prev_time = 0.0
    while viewer.is_running():
        step_start = time.time()
        # Detect MuJoCo reset and reset variables
        if data.time < prev_time:
            tipped = False
            mujoco.mj_forward(model, data)
            pitch = 0.0
        prev_time = data.time
        # get the error from the robots orientation
        # since we want the robot to stay upright, the error is the difference between pitch and 0.0
        gyro = data.sensor(IMU_GYRO).data
        # integrate the gyro reading to get the pitch angle (assuming small angles)
        pitch += gyro[1] * model.opt.timestep  # Assuming gyro[1] is the pitch rate
        error = pitch  # Desired pitch is 0.0
        if abs(pitch) > 0.5:  # If the robot tips over
            tipped = True

        # Write the latest ctrl values into the sim.
        if not tipped:
            control_action = pid.update(error=error,error_rate=gyro[1], dt=model.opt.timestep)  # Replace 0.0 with actual error
        else:
            control_action = 0.0  # Stop the robot if it has tipped over
        data.ctrl[left_motor_id] = control_action
        data.ctrl[right_motor_id] = control_action

        # Advance simulation by one step (determined by timestep attribute in MJCF file)
        mujoco.mj_step(model, data)

        # Render the current simulation state
        viewer.sync()

        # Print the sensor readings every PRINT_EVERY steps
        steps += 1
        if steps % PRINT_EVERY == 0:
            accel = data.sensor(IMU_ACCEL).data
            gyro = data.sensor(IMU_GYRO).data
            orientation = data.sensor(IMU_ORIENTATION).data
            left_pos = data.sensor(LEFT_WHEEL_POS).data
            right_pos = data.sensor(RIGHT_WHEEL_POS).data
            left_vel = data.sensor(LEFT_WHEEL_VEL).data
            right_vel = data.sensor(RIGHT_WHEEL_VEL).data

            # Clear output before printing so we don't flood it
            clear_output(wait=True)
            print("current error:", error)
            print("control:", control_action)

        # Make sure the loop iteration time matches the MJCF timestep time (2 ms)
        # i.e. slow the simulation down so we can see the robot behave in real time
        slack = model.opt.timestep - (time.time() - step_start)
        if slack > 0:
            time.sleep(slack)

current error: 0.00020676967265440097
control: -0.029445847548149877


KeyboardInterrupt: 